# Superstore Data Cleaning

This notebook profiles the Sample Superstore dataset, checks missing values and duplicate rows, removes exact duplicate records, and exports a cleaned CSV.

In [2]:
%cd /content

!if [ ! -d /content/Superestore/.git ]; then \
  git clone --depth 1 --branch portfolio/superstore-notebook-reproducibility \
  https://github.com/Henzo360/Superestore.git /content/Superestore; \
fi

%cd /content/Superestore

!ls -lh *.csv

/content
Cloning into '/content/Superestore'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 1.04 MiB | 7.12 MiB/s, done.
/content/Superestore
-rw-r--r-- 1 root root 1.1M Oct 26 21:20 SampleSuperstore.csv


In [3]:
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [Path("data") / "SampleSuperstore.csv", Path("SampleSuperstore.csv")]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])
OUTPUT_PATH = Path("output") / "sample_superstore_cleaned.csv"

if not DATA_PATH.exists():
    expected_paths = ", ".join(str(path) for path in DATA_CANDIDATES)
    raise FileNotFoundError(f"Could not find SampleSuperstore.csv. Checked: {expected_paths}")

In [4]:
sample = pd.read_csv(DATA_PATH)
print(f"Loaded dataset shape: {sample.shape}")

Loaded dataset shape: (9994, 13)


## Inspect the dataset

Review a sample of rows, the dataset dimensions, data types, and non-null counts before cleaning.

In [5]:
display(sample.head())

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
0,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,261.9600,2,0.00,41.9136
1,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,731.9400,3,0.00,219.5820
2,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,14.6200,2,0.00,6.8714
3,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,957.5775,5,0.45,-383.0310
4,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,22.3680,2,0.20,2.5164


In [6]:
print('Dataset shape:', sample.shape)

Dataset shape: (9994, 13)


In [7]:
sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Ship Mode     9994 non-null   object 
 1   Segment       9994 non-null   object 
 2   Country       9994 non-null   object 
 3   City          9994 non-null   object 
 4   State         9994 non-null   object 
 5   Postal Code   9994 non-null   int64  
 6   Region        9994 non-null   object 
 7   Category      9994 non-null   object 
 8   Sub-Category  9994 non-null   object 
 9   Sales         9994 non-null   float64
 10  Quantity      9994 non-null   int64  
 11  Discount      9994 non-null   float64
 12  Profit        9994 non-null   float64
dtypes: float64(3), int64(2), object(8)
memory usage: 1015.1+ KB


## Check missing values

Missing-value counts are reported by column. A missing value should be assessed in context; it is not automatically proof that a record should be deleted.

In [8]:
missing_counts = sample.isna().sum()
display(missing_counts[missing_counts.gt(0)].sort_values(ascending=False).to_frame("missing_count"))
if not missing_counts.gt(0).any():
    print("No missing values detected.")

,missing_count


No missing values detected.


Use the report above to identify any columns with missing values. The notebook does not claim that the dataset is complete unless the check confirms it.

## Check and remove duplicate rows

First count exact duplicate rows and display them for inspection. Then remove exact duplicates and compare the row count before and after.

In [9]:
duplicates_count = int(sample.duplicated().sum())
print(f"Exact duplicate rows: {duplicates_count:,}")

Exact duplicate rows: 17


In [10]:
duplicate_mask = sample.duplicated(keep=False)
if duplicate_mask.any():
    sort_columns = ["Postal Code"] if "Postal Code" in sample.columns else list(sample.columns[:1])
    display(sample.loc[duplicate_mask].sort_values(sort_columns))
else:
    print("No duplicate rows to display.")

,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub-Category,Sales,Quantity,Discount,Profit
3127,Standard Class,Consumer,United States,New York City,New York,10011,East,Office Supplies,Paper,49.120,4,0.0,23.0864
6334,Standard Class,Consumer,United States,New York City,New York,10011,East,Office Supplies,Paper,49.120,4,0.0,23.0864
935,Standard Class,Home Office,United States,Philadelphia,Pennsylvania,19120,East,Office Supplies,Paper,15.552,3,0.2,5.4432
950,Standard Class,Home Office,United States,Philadelphia,Pennsylvania,19120,East,Office Supplies,Paper,15.552,3,0.2,5.4432
3406,Standard Class,Home Office,United States,Columbus,Ohio,43229,East,Furniture,Chairs,281.372,2,0.3,-12.0588
3405,Standard Class,Home Office,United States,Columbus,Ohio,43229,East,Furniture,Chairs,281.372,2,0.3,-12.0588
8533,Standard Class,Consumer,United States,Detroit,Michigan,48227,Central,Furniture,Chairs,389.970,3,0.0,35.0973
9262,Standard Class,Consumer,United States,Detroit,Michigan,48227,Central,Furniture,Chairs,389.970,3,0.0,35.0973
8457,Second Class,Corporate,United States,Chicago,Illinois,60653,Central,Office Supplies,Binders,3.564,3,0.8,-6.2370
9477,Second Class,Corporate,United States,Chicago,Illinois,60653,Central,Office Supplies,Binders,3.564,3,0.8,-6.2370


In [11]:
rows_before = len(sample)
sample_df = sample.drop_duplicates().copy()
rows_removed = rows_before - len(sample_df)
print(f"Rows before cleaning: {rows_before:,}")
print(f"Exact duplicate rows removed: {rows_removed:,}")
print(f"Rows after cleaning: {len(sample_df):,}")

Rows before cleaning: 9,994
Exact duplicate rows removed: 17
Rows after cleaning: 9,977


Exact duplicate rows are removed with `drop_duplicates()`. The printed counts above report the actual number removed from the input file.

## Export the cleaned dataset

In [12]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample_df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned CSV saved to: {OUTPUT_PATH.resolve()}")

Cleaned CSV saved to: /content/Superestore/output/sample_superstore_cleaned.csv
